In [4]:
import torch
from omegaconf import OmegaConf
from pathlib import Path
from einops import rearrange
from functools import partial
import sys

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.models.autoencoder import get_autoencoder


ModuleNotFoundError: No module named 'xformers'

In [ ]:



from lola.autoencoder import get_autoencoder
from lola.data import field_preprocess, get_well_inputs, get_well_multi_dataset
from lola.plot import plot_fields, plot_psd

torch.set_float32_matmul_precision("high")

device = torch.device("cuda")
rng = torch.manual_seed(0)

In [ ]:
# runpath = Path("~/ceph/lola/runs/ae/8brbetky_rayleigh_benard_dcae_f32c4_large/")
# runpath = Path("~/ceph/lola/runs/ae/jiof10wl_rayleigh_benard_dcae_f32c16_large/")
runpath = Path("~/ceph/lola/runs/ae/1e3z5x2c_rayleigh_benard_dcae_f32c64_large/")
runpath = runpath.expanduser()

cfg = OmegaConf.load(runpath / "config.yaml")

## Data

In [ ]:
dataset = get_well_multi_dataset(
    path="~/ceph/the_well/datasets",
    physics=cfg.dataset.physics,
    split="valid",
    steps=1,
    include_filters=cfg.dataset.include_filters,
)

preprocess = partial(
    field_preprocess,
    mean=torch.as_tensor(cfg.dataset.stats.mean, device=device),
    std=torch.as_tensor(cfg.dataset.stats.std, device=device),
    transform=cfg.dataset.transform,
)

## Autoencoder

In [ ]:
autoencoder = get_autoencoder(**cfg.ae)
autoencoder.load_state_dict(torch.load(runpath / "state.pth", weights_only=True, map_location=device))
autoencoder.to(device)
autoencoder.eval()

In [ ]:
sum(p.numel() for p in autoencoder.parameters())

## Evaluation

In [ ]:
x, _ = get_well_inputs(dataset[6666], device=device)
x = preprocess(x)
x = rearrange(x, "1 H W C -> C H W")

In [ ]:
with torch.no_grad():
    z = autoencoder.encode(x.unsqueeze(0))
    y = autoencoder.decode(z).squeeze(0)

error = (x - y).abs().reshape(dataset.metadata.n_fields, -1)

In [ ]:
(
    error.mean(axis=-1),
    error.square().mean(axis=-1).sqrt(),
    error.median(axis=-1).values,
    error.max(axis=-1).values,
)

In [ ]:
fig = plot_fields(x[:, None].mT, y[:, None].mT, fields=cfg.dataset.fields, tolerance=1.0, figsize=(4.8, 1.6))

In [ ]:
fig = plot_psd(x, y, fields=cfg.dataset.fields)